In [29]:
import pandas as pd
import re
from collections import defaultdict 

In [31]:
data_file = "C:\\Users\\shirl\\Documents\\Studie\\2025-2026\\Thesis\\personalized-coping-challenges\\data\\Coping challenge beoordelingen_May 18, 2026_15.59.csv"
df = pd.read_csv(data_file)

challenge_ids_file = "C:\\Users\\shirl\\Documents\\Studie\\2025-2026\\Thesis\\personalized-coping-challenges\\data\\challenge_ids.csv"
challenge_ids_df = pd.read_csv(challenge_ids_file, delimiter=';')


In [21]:
label_to_value = {
    "Geen bijdrage": 0,
    "Kleine bijdrage": 1,
    "Veel bijdrage": 2
}

categories = ['acceptation', 'distraction', 'problem_solving', 'social_support']

In [33]:
participant_col = "ResponseId"

pattern = re.compile(r"^(Q\d+)_(\d)$")

rows = []

for _, row in df.iterrows():

    participant_id = row[participant_col]

    challenges = {}

    for col in df.columns:
        match = pattern.match(col)
        if match:
            challenge = match.group(1)
            rating_num = match.group(2)     
            category = categories[int(rating_num) - 1]  # Map to category

            if challenge not in challenges:
                challenges[challenge] = {
                    "participant_id": participant_id,
                    "question": challenge,
                    "challenge": challenge_ids_df.loc[challenge_ids_df['challenge'] == challenge, 'challenge_id'].values[0]
                }

            challenges[challenge][category] = label_to_value.get(row[col], 0)

    # Add all challenge rows for this participant
    rows.extend(challenges.values())

# Final dataframe
result = pd.DataFrame(rows)

num_challenges = result['challenge'].nunique()
print(f"Total unique challenges: {num_challenges}")

result.head()

Total unique challenges: 103


,participant_id,question,challenge,acceptation,distraction,problem_solving,social_support
0,R_2jpoNvYmT6pfujq,Q1,PO17,0,0,2,0
1,R_2jpoNvYmT6pfujq,Q2,PO9,0,1,1,2
2,R_2jpoNvYmT6pfujq,Q12,AF19,0,2,0,0
3,R_2jpoNvYmT6pfujq,Q13,AC15,1,1,0,0
4,R_2jpoNvYmT6pfujq,Q14,PO5,0,0,2,0


In [35]:
# obtain single score for each category per challenge by averaging across challenges for each participant
final_df = result.groupby(['challenge'])[categories].mean().reset_index()
display(final_df.head())


,challenge,acceptation,distraction,problem_solving,social_support
0,AC1,1.0,1.0,0.0,0.0
1,AC10,2.0,0.0,0.0,0.0
2,AC11,1.0,0.0,0.0,2.0
3,AC12,1.0,1.0,0.0,0.0
4,AC13,0.0,2.0,0.0,0.0
